### 0. Import modules

In [ ]:
import stable_baselines3 as sb3
import gym_unbalanced_disk
import gymnasium as gym
import numpy as np
np.random.seed(42)
sb3.common.utils.set_random_seed(42)

### 1 Initializing the saving dirs, Prepare the Env, prepare the callback funcitons

In [52]:
"""
Train an A2C/PPO agent on the unbalanced-disk swing-up task using stable-baselines3.

Run from anywhere (the gym_unbalanced_disk import registers the env id):
    python a2c.py

View the training logs with:
    tensorboard --logdir ./tensorboard_logs
"""

import time
from pathlib import Path

import numpy as np
import gymnasium as gym

# Importing gym_unbalanced_disk registers the 'unbalanced-disk-v0' env id with
# gymnasium, so it must be imported before any gym.make(...) call.
import gym_unbalanced_disk  # noqa: F401  (imported for its registration side effect)
import stable_baselines3 as sb3
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    CallbackList,
    BaseCallback,
)


def get_save_dirs(model_name: str):
    # All artifacts (checkpoints, logs) are written next to this script, regardless
    # of the directory the script is launched from.
    try:
        HERE = Path(__file__).resolve().parent
    except NameError:
        # __file__ is undefined in a Jupyter notebook; use the working dir
        HERE = Path.cwd()
    TB_LOG_DIR = HERE / "tensorboard_logs"
    BEST_MODEL_DIR = HERE / f"{model_name}_best"
    EVAL_LOG_DIR = HERE / f"{model_name}_eval_logs"
    CHECKPOINT_DIR = HERE / f"{model_name}_checkpoints"
    FINAL_MODEL_PATH = HERE / f"{model_name}_unbalanced_disk"
    return TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH


# ---------------------------------------------------------------------------
# Environment factory
# ---------------------------------------------------------------------------
def make_env():
    """
    Build a single training/eval environment.
    """
    env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0)
    env = Monitor(env)
    return env


# ---------------------------------------------------------------------------
# Standalone evaluation (used for a final report after training)
# ---------------------------------------------------------------------------
def evaluate(model, env, num_episodes=10):
    """
    Run the (deterministic) policy for a number of episodes and return the
    mean total reward per episode.
    """
    total_rewards = []
    for _ in range(num_episodes):
        obs, info = env.reset()
        done = False
        episode_reward = 0.0
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            reward = env.evaluation_reward_fun()
            episode_reward += reward
            done = terminated or truncated
        total_rewards.append(episode_reward)
    return float(np.mean(total_rewards))

def evaluate_trained_policy(model_path: Path, num_episodes=10):
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")
    eval_env = make_env()
    mean_reward = evaluate(model, eval_env, num_episodes=num_episodes)
    print(f"Mean reward over {num_episodes} episodes: {mean_reward:.2f}")
    eval_env.close()

def visualize_trained_policy(model_path: Path, deterministic: bool = False):
# -----------------------------------------------------------------------
# Visualise the trained policy
# -----------------------------------------------------------------------
# A human-rendered env to watch the swing-up. render_mode="human" opens a
# pygame window.
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")
    vis_env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0, render_mode="human")
    obs, info = vis_env.reset()
    try:
        for _ in range(200):
            action, _states = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = vis_env.step(action)
            vis_env.render()
            time.sleep(1 / 40)
            if terminated or truncated:
                obs, info = vis_env.reset()   # unpack the (obs, info) tuple
    finally:  # always run, even on Ctrl-C, so the window/resources are released
        vis_env.close()


class RenderEvalCallback(BaseCallback):
    """Every `render_freq` steps, play one deterministic episode in a
    human-rendered env so the current policy can be watched. Kept separate
    from EvalCallback so metric logging (eval_freq) stays untouched."""
    def __init__(self, render_freq=50000, n_episodes=1, max_steps=200, verbose=0):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.n_episodes = n_episodes
        self.max_steps = max_steps

    def _on_step(self) -> bool:
        # n_calls increments once per _on_step; with a single env this == timesteps.
        # For multiple envs use self.num_timesteps instead.
        if self.n_calls % self.render_freq == 0:
            vis_env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0,
                               render_mode="human")
            try:
                for _ in range(self.n_episodes):
                    obs, info = vis_env.reset()
                    for _ in range(self.max_steps):
                        action, _ = self.model.predict(obs, deterministic=True)
                        obs, reward, terminated, truncated, info = vis_env.step(action)
                        vis_env.render()
                        time.sleep(1 / 24)
                        if terminated or truncated:
                            break
            finally:
                vis_env.close()
        return True


def get_callbacks(eval_env):


    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(BEST_MODEL_DIR),
        log_path=str(EVAL_LOG_DIR),
        eval_freq=25000,       # run an evaluation every 10k training steps
        n_eval_episodes=5,
        deterministic=True,
        render=False,
    )

    # Metric eval every 20k steps + a watchable rendered rollout every 50k steps.
    callbacks = CallbackList([eval_callback, RenderEvalCallback(render_freq=200000)])

    return callbacks

## 2. Running experiments

 the models for the Advanced RL chapter of the report. The current reward structure in UnbalancedDisc.py represents the Tuned_R from the rapport, therefor running the following cells wont reproduce experiment 1 unless the reward function is explicitely changed.


### 2.1  Experiment 1/2:

#### 2.1.1 A2C training
Below is a script that allows for base training of the experiment 1 and 2 A2C models

In [ ]:

# Separate environments for training and for periodic evaluation, so eval
# episodes do not interfere with the training rollouts.
train_env = make_env()
eval_env = make_env()
TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("a2c_tuned_reward")

def train_a2c():
    # A2C with the default MLP policy. device="cpu" is correct here: the network
    # is tiny and the bottleneck is the scipy ODE solve inside each env step,
    # so a GPU would not help.
    model = sb3.A2C(
        "MlpPolicy",
        train_env,
        verbose=0,
        device="cpu",
        tensorboard_log=str(TB_LOG_DIR),
    )

    callbacks = get_callbacks(eval_env)

    # Train. progress_bar=True needs the 'tqdm' and 'rich' packages
    # (set it to False if they are not installed).
    model.learn(
        total_timesteps=1000000,
        tb_log_name="a2c_base_reward",
        progress_bar=True,
    )

    # Save the final (last) model. The best model seen during training was
    # already saved by EvalCallback at BEST_MODEL_DIR / "best_model.zip".
    model.save(str(FINAL_MODEL_PATH))
    print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
    print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

    # Prefer the best checkpoint for evaluation/visualisation; fall back to the
    # final model if (e.g. for a very short run) no eval ever triggered.
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.A2C.load(str(best_model_path), device="cpu")
        print("Loaded best model for evaluation.")

    # Final evaluation report on a fresh eval env.
    mean_reward = evaluate(model, eval_env, num_episodes=10)
    print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")

    train_env.close()
    eval_env.close()

#train_a2c()
#evaluate_trained_policy(Path("a2c_base_reward_unbalanced_disk.zip"))
visualize_trained_policy(Path("a2c_base_reward_unbalanced_disk.zip"))


#### 2.1.2 PPO training cell

In [38]:
TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("ppo_tuned_reward")
train_env = make_env()
eval_env = make_env()

def train_ppo():
    model = sb3.PPO(
        "MlpPolicy",
        train_env,
        verbose=0,
        device="cpu",
        tensorboard_log=str(TB_LOG_DIR),
    )


    callbacks = get_callbacks(eval_env)
    model.learn(
            total_timesteps=1000000,
            tb_log_name="ppo_base_reward",
            progress_bar=True,
        )
    # Save the final (last) model. The best model seen during training was
    # already saved by EvalCallback at BEST_MODEL_DIR / "best_model.zip".
    model.save(str(FINAL_MODEL_PATH))
    print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
    print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

    # Prefer the best checkpoint for evaluation/visualisation; fall back to the
    # final model if (e.g. for a very short run) no eval ever triggered.
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.PPO.load(str(best_model_path), device="cpu")
        print("Loaded best model for evaluation.")

    # Final evaluation report on a fresh eval env.
    mean_reward = evaluate(model, eval_env, num_episodes=10)
    print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")


    train_env.close()
    eval_env.close()

    visualize_trained_policy( Path("ppo_tuned_reward_unbalanced_disk.zip"))

#train_ppo()
best_model_path = Path("ppo_tuned_reward_unbalanced_disk.zip")

evaluate_trained_policy(best_model_path)
visualize_trained_policy(best_model_path)

Mean reward over 10 episodes: 172.22


### 2.2 Experiment 3: Hyperparam tuning

After running HPO.py we found the best hyperparams for both A2C and PPO. The cells below train both models on their best params

### 2.2.1  Optimal A2C Training

In [43]:
# --- Train A2C with the best hyperparameters found by the Optuna sweep (hpo.py) ---
# Reads hpo/a2c/best_params.json so this stays in sync with the latest sweep;
# re-run hpo.py and the params here update automatically.
import json
from pathlib import Path

def train_a2c_tuned():

    NET_ARCHS = {"small": [64, 64], "medium": [256, 256]}
    best = json.loads((Path.cwd() / "hpo" / "a2c" / "best_params.json").read_text())
    params = dict(best["params"])
    # net_arch is stored as a label ("small"/"medium"); expand it back to a net spec.
    params["policy_kwargs"] = dict(net_arch=NET_ARCHS[params.pop("net_arch")])
    print(f"Best sweep reward: {best['value']:.2f}")
    for k, v in params.items():
        print(f"  {k}: {v}")

    # Separate dirs so the tuned run does not overwrite the earlier a2c_best/ etc.
    TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("a2c_tuned_reward_HPO")
    train_env = make_env()
    eval_env = make_env()
    callbacks = get_callbacks(eval_env)

    model = sb3.A2C(
        "MlpPolicy",
        train_env,
        device="cpu",
        verbose=0,
        tensorboard_log=str(TB_LOG_DIR),
        **params,
    )

    model.learn(
        total_timesteps=1_000_000,
        tb_log_name="a2c_tuned_reward_HPO",
        progress_bar=True,
    )

    model.save(str(FINAL_MODEL_PATH))
    print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
    print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

    # Prefer the best checkpoint for the final report / visualisation.
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.A2C.load(str(best_model_path), device="cpu")
        print("Loaded best model for evaluation.")

    mean_reward = evaluate(model, eval_env, num_episodes=10)
    print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")

    train_env.close()
    eval_env.close()
#train_a2c_tuned()
best_model_path = Path("a2c_tuned_reward_HPO_unbalanced_disk.zip")
visualize_trained_policy(best_model_path)

#### 2.2.2 Optimal PPO training

In [45]:
# --- Train PPO with the best hyperparameters found by the Optuna sweep (hpo.py) ---
# Reads hpo/ppo/best_params.json so this stays in sync with the latest sweep;
# re-run hpo.py and the params here update automatically.
import json
from pathlib import Path

def train_ppo_tuned():
    NET_ARCHS = {"small": [64, 64], "medium": [256, 256]}

    best = json.loads((Path.cwd() / "hpo" / "ppo" / "best_params.json").read_text())
    params = dict(best["params"])
    # net_arch is stored as a label ("small"/"medium"); expand it back to a net spec.
    params["policy_kwargs"] = dict(net_arch=NET_ARCHS[params.pop("net_arch")])
    print(f"Best sweep reward: {best['value']:.2f}")
    for k, v in params.items():
        print(f"  {k}: {v}")

    # Separate dirs so the tuned run does not overwrite the earlier ppo_best/ etc.
    TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("ppo_tuned_reward_HPO")
    train_env = make_env()
    eval_env = make_env()
    callbacks = get_callbacks(eval_env)

    model = sb3.PPO(
        "MlpPolicy",
        train_env,
        device="cpu",
        verbose=0,
        tensorboard_log=str(TB_LOG_DIR),
        **params,
    )

    model.learn(
        total_timesteps=1_000_000,
        tb_log_name="ppo_tuned_reward_HPO",
        progress_bar=True,
    )

    model.save(str(FINAL_MODEL_PATH))
    print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
    print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

    # Prefer the best checkpoint for the final report / visualisation.
    best_model_path = BEST_MODEL_DIR / "best_model.zip"
    if best_model_path.exists():
        model = sb3.PPO.load(str(best_model_path), device="cpu")
        print("Loaded best model for evaluation.")

    mean_reward = evaluate(model, eval_env, num_episodes=10)
    print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")

    train_env.close()
    eval_env.close()
train_ppo_tuned()

Best sweep reward: 270.97

n_steps: 256

batch_size: 64

n_epochs: 12

gamma: 0.95

...

gae_lambda: 0.8

max_grad_norm: 0.5

policy_kwargs: {'net_arch': [256, 256]}

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1,000,192/1,000,000  [ 0:13:40 < 0:00:00 , 1,179 it/s ]

Final model saved to: /home/nemo/Documents/GitHub/IML/DesignProject/DesignProject/Part2_Control/A2C/ppo_tuned_reward_HPO_unbalanced_disk.zip
Best model saved to : /home/nemo/Documents/GitHub/IML/DesignProject/DesignProject/Part2_Control/A2C/ppo_tuned_reward_HPO_best/best_model.zip

Final mean reward over 10 episodes: 170.27


In [53]:
from pathlib import Path

best_model_path = Path("ppo_tuned_reward_HPO_unbalanced_disk.zip")
visualize_trained_policy(best_model_path)